# Install kaggle-environments

In [14]:
# 1. Enable Internet in the Kernel (Settings side pane)

# 2. Curl cache may need purged if v0.1.6 cannot be found (uncomment if needed). 
# !curl -X PURGE https://pypi.org/simple/kaggle-environments

# ConnectX environment was defined in v0.1.6
# !pip install kaggle-environments>=0.1.6

# Create ConnectX Environment

In [1]:
from kaggle_environments import evaluate, make, utils

env = make("connectx", debug=True)
env.render()

+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+



In [2]:
env.specification["reward"]

{'description': '0 = Lost, 0.5 = Draw, 1 = Won',
 'enum': [0, 0.5, 1],
 'default': 0.5,
 'type': ['number', 'null']}

In [20]:
env.specification["observation"]

{'board': {'description': 'Serialized grid (rows x columns). 0 = Empty, 1 = P1, 2 = P2',
  'type': 'array',
  'default': []},
 'mark': {'default': 0,
  'description': 'Which checkers are the agents.',
  'enum': [1, 2]}}

In [25]:
env.specification["action"]

{'description': 'Column to drop a checker onto the board.',
 'type': 'integer',
 'minimum': 0,
 'default': 0}

In [11]:
env.specification["info"]

{}

In [24]:
env.interpreter

<function kaggle_environments.envs.connectx.connectx.interpreter(state, env)>

In [13]:
env.state

[{'action': 0,
  'reward': 0.5,
  'info': {},
  'observation': {'board': [0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0],
   'mark': 1},
  'status': 'ACTIVE'},
 {'action': 0,
  'reward': 0.5,
  'info': {},
  'observation': {'board': [0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0],
   'mark': 2},
  'status': 'INACTIVE'}]

In [21]:
env.observation.board

AttributeError: 'Environment' object has no attribute 'observation'

# Create an Agent

To create the submission, an agent function should be fully encapsulated (no external dependencies).  

When your agent is being evaluated against others, it will not have access to the Kaggle docker image.  Only the following can be imported: Python Standard Library Modules, gym, numpy, scipy, pytorch (1.3.1, cpu only), and more may be added later.



In [3]:
# This agent random chooses a non-empty column.
def my_agent(observation, configuration):
    from random import choice
    return choice([c for c in range(configuration.columns) if observation.board[c] == 0])

In [5]:
#
# Agent plus intelligent : essaie de gagner ou de bloquer l'adversaire
def smart_agent(observation, configuration):
    import numpy as np
    
    def drop_piece(board, col, mark, config):
        board_copy = board.copy()
        for row in range(config.rows-1, -1, -1):
            if board_copy[row * config.columns + col] == 0:
                board_copy[row * config.columns + col] = mark
                return board_copy
        return board_copy  # Si colonne pleine, mais on ne devrait pas arriver là
    
    def check_win(board, config):
        # Vérifie les victoires horizontales, verticales, diagonales
        rows = config.rows
        cols = config.columns
        inarow = config.inarow  # Généralement 4 pour Connect4
        
        # Horizontal
        for r in range(rows):
            for c in range(cols - inarow + 1):
                if all(board[r*cols + c + i] == board[r*cols + c] and board[r*cols + c] != 0 for i in range(inarow)):
                    return True
        
        # Vertical
        for r in range(rows - inarow + 1):
            for c in range(cols):
                if all(board[(r+i)*cols + c] == board[r*cols + c] and board[r*cols + c] != 0 for i in range(inarow)):
                    return True
        
        # Diagonale \
        for r in range(rows - inarow + 1):
            for c in range(cols - inarow + 1):
                if all(board[(r+i)*cols + (c+i)] == board[r*cols + c] and board[r*cols + c] != 0 for i in range(inarow)):
                    return True
        
        # Diagonale /
        for r in range(rows - inarow + 1):
            for c in range(inarow - 1, cols):
                if all(board[(r+i)*cols + (c-i)] == board[r*cols + c] and board[r*cols + c] != 0 for i in range(inarow)):
                    return True
        
        return False
    
    board = observation.board
    columns = configuration.columns
    rows = configuration.rows
    mark = observation.mark
    
    # Liste des colonnes valides
    valid_moves = [c for c in range(columns) if board[c] == 0]
    
    # Essaie de gagner
    for col in valid_moves:
        new_board = drop_piece(board, col, mark, configuration)
        if check_win(new_board, configuration):
            return col
    
    # Essaie de bloquer l'adversaire
    opponent_mark = 3 - mark
    for col in valid_moves:
        new_board = drop_piece(board, col, opponent_mark, configuration)
        if check_win(new_board, configuration):
            return col
    
    # Sinon, joue au centre ou aléatoirement
    center = columns // 2
    if center in valid_moves:
        return center
    import random
    return random.choice(valid_moves)

In [15]:
# Créer un agent avec Reinforcement Learning
from stable_baselines3 import PPO
import gymnasium as gym
from gymnasium import spaces
import numpy as np

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [ ]:
# Wrapper pour rendre l'environnement compatible avec Gym
class ConnectXEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.env = make("connectx", debug=True)
        self.action_space = spaces.Discrete(self.env.configuration.columns)  # Actions: 0 à columns-1
        self.observation_space = spaces.MultiDiscrete([3] * (self.env.configuration.rows * self.env.configuration.columns))  # Board: 0,1,2
        self.trainer = elf.env.train([None, "random"])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        return self.trainer.reset()

    def step(self, action):
        try:
            obs, reward, done, info = self.trainer.step(int(action))
            if reward is None:
                reward = 0  # Pas de récompense intermédiaire
            return np.array(obs.board, dtype=int), float(reward), done, done, info  # Gymnasium format
        except Exception as e:
            # Action invalide, pénaliser
            obs = self.trainer.reset()  # Reset ou continuer
            return np.array(obs.board, dtype=int), -10, True, True, {"error": str(e)}  # Pénalité pour action invalide
    
    def render(self, **kwargs):
        return self.env.render(**kwargs)

# Créer l'env wrapper
connectx_env = ConnectXEnv(env)

# Entraîner avec PPO
model = PPO("MlpPolicy", connectx_env, verbose=1)
model.learn(total_timesteps=10000)  # Entraîner pendant 10k steps

# Sauvegarder le modèle
model.save("ppo_connectx")

print("Modèle RL entraîné et sauvegardé !")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 0
Timeout: Timed out after 2 seconds.
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 1
Timeout: Timed out after 2 seconds.
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 0
Timeout: Timed out after 2 seconds.
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 0
Timeout: Timed out after 2 seconds.
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 1
Timeout: Timed out after 2 seconds.
Invalid Action: Invalid column: 0
Invalid Action: Invalid column: 3
Invalid Acti

In [ ]:
# Créer un agent RL basé sur le modèle entraîné
def rl_agent(observation, configuration):
    # Charger le modèle (si pas déjà chargé)
    global rl_model
    if 'rl_model' not in globals():
        try:
            rl_model = PPO.load("ppo_connectx")
        except Exception as e:
            print(f"Erreur lors du chargement du modèle: {e}")
            rl_model = None
    
    # Préparer l'observation
    obs = np.array(observation.board, dtype=int).reshape(1, -1)
    
    # Prédire l'action
    try:
        if rl_model is not None:
            action, _ = rl_model.predict(obs, deterministic=True)
            action = int(action.item())
        else:
            action = -1  # Forcer le fallback
    except Exception as e:
        print(f"Erreur lors de la prédiction: {e}")
        action = -1
    
    # Vérifier si valide
    valid_moves = [c for c in range(configuration.columns) if observation.board[c] == 0]
    if action in valid_moves:
        return action
    else:
        # Sinon, choisir aléatoirement
        import random
        return random.choice(valid_moves) if valid_moves else 0

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Model(nn.Model):
    def __init__(self, hidden_size=128):
        super().__init__()
        self.conv1 = nn.Conv2d(2, hidden_size, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(hidden_size)
        self.conv2 = nn.Conv2d(hidden_size, hidden_size, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(hidden_size)
        self.conv3 = nn.Conv2d(hidden_size, hidden_size, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(hidden_size)
        self.ln = nn.Linear(hidden_size * 6 * 7, 7)
        self.output = nn.Softmax(dim=1)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = x.view(-1, self.num_flat_features(x))
        x = self.ln(x)
        x = self.output(x)
        return x



# Test your Agent

In [4]:
env.reset()
# Play as the first agent against default "random" agent.
env.run([my_agent, "random"])
env.render(mode="ipython", width=500, height=450)

In [ ]:
#
# Test de l'agent intelligent
env.reset()
env.run([smart_agent, "random"])
env.render(mode="ipython", width=500, height=450)

In [ ]:
# Afficher le board après la partie
print("Board après la partie :", env.state[0].observation.board)
print("Configuration : rows =", env.configuration.rows, ", columns =", env.configuration.columns)

In [ ]:
# Tester l'agent RL
env.reset()
env.run([rl_agent, "random"])
env.render(mode="ipython", width=500, height=450)

# Debug/Train your Agent

In [6]:
# Play as first position against random agent.
trainer = env.train([None, "random"])

observation = trainer.reset()

while not env.done:
    my_action = my_agent(observation, env.configuration)
    print("My Action", my_action)
    observation, reward, done, info = trainer.step(my_action)
    # env.render(mode="ipython", width=100, height=90, header=False, controls=False)
env.render()

My Action 1
My Action 3
My Action 6
My Action 2
+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 2 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 2 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 2 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 2 | 1 | 1 | 1 | 0 | 0 | 1 |
+---+---+---+---+---+---+---+



# Evaluate your Agent

In [7]:
def mean_reward(rewards):
    return sum(r[0] for r in rewards) / float(len(rewards))

# Run multiple episodes to estimate its performance.
print("My Agent vs Random Agent:", mean_reward(evaluate("connectx", [my_agent, "random"], num_episodes=10)))
print("My Agent vs Negamax Agent:", mean_reward(evaluate("connectx", [my_agent, "negamax"], num_episodes=10)))

My Agent vs Random Agent: 0.8
My Agent vs Negamax Agent: 0.05


In [10]:
# Run multiple episodes to estimate its performance.
print("Smart Agent vs Random Agent:", mean_reward(evaluate("connectx", [smart_agent, "random"], num_episodes=10)))
print("Smart Agent vs Negamax Agent:", mean_reward(evaluate("connectx", [smart_agent, "negamax"], num_episodes=10)))

Smart Agent vs Random Agent: 1.0
Smart Agent vs Negamax Agent: 0.85


# Play your Agent
Click on any column to place a checker there ("manually select action").

In [20]:
# "None" represents which agent you'll manually play as (first or second player).
env.play([None, "negamax"], width=500, height=450)

AttributeError: 'Environment' object has no attribute 'play'

# Write Submission File



In [ ]:
import inspect
import os

def write_agent_to_file(function, file):
    with open(file, "a" if os.path.exists(file) else "w") as f:
        f.write(inspect.getsource(function))
        print(function, "written to", file)

write_agent_to_file(my_agent, "submission.py")

<function smart_agent at 0x7d1a913bc9a0> written to submission.py


# Validate Submission
Play your submission against itself.  This is the first episode the competition will run to weed out erroneous agents.

Why validate? This roughly verifies that your submission is fully encapsulated and can be run remotely.

In [22]:
# Note: Stdout replacement is a temporary workaround.
import sys
out = sys.stdout
submission = utils.read_file("/kaggle/working/submission.py")
agent = utils.get_last_callable(submission)
sys.stdout = out

env = make("connectx", debug=True)
env.run([agent, agent])
print("Success!" if env.state[0].status == env.state[1].status == "DONE" else "Failed...")

NotFound: /kaggle/working/submission.py not found

# Submit to Competition

1. Commit this kernel.
2. View the commited version.
3. Go to "Data" section and find submission.py file.
4. Click "Submit to Competition"
5. Go to [My Submissions](https://kaggle.com/c/connectx/submissions) to view your score and episodes being played.